# Vector storage adapters

Both adapters implement `BaseVectorStorage`, so RAGU can swap them without the
pipeline noticing. They are not equivalent:

- **`NanoVectorDBStorage`** is a dependency-free numpy store persisted to a JSON
  file. It is the default, fine up to tens of thousands of vectors, and
  **dense-only**: sparse embeddings passed to it are logged and dropped.
- **`QdrantVectorDBStorage`** runs in-memory, on-disk or against a remote server,
  and is the only adapter that supports sparse vectors and therefore hybrid
  retrieval.

This notebook drives both at the level RAGU's `Index` uses them: `Point` in,
`EmbeddingHit` out.

**Environment:** `OPENAI_API_KEY`, `EMBEDDER_MODEL_NAME`, and optionally
`OPENAI_BASE_URL`.

In [ ]:
import os
import shutil
import tempfile
from pathlib import Path

import numpy as np

from ragu.models.embedder import EmbedderOpenAI
from ragu.models.openai import CachedAsyncOpenAI
from ragu.models.sparse_embedder import BM25
from ragu.storage.types import EmbeddingHit, Point
from ragu.storage.vdb_storage_adapters.nano_vdb import NanoVectorDBStorage
from ragu.storage.vdb_storage_adapters.qdrant_vdb import QdrantVectorDBStorage

DOCUMENTS = [
    ("doc-1", "Dennis Ritchie created the C programming language at Bell Labs."),
    ("doc-2", "Ken Thompson and Dennis Ritchie developed the Unix operating system."),
    ("doc-3", "Guido van Rossum created Python in the late 1980s."),
    ("doc-4", "The Eiffel Tower is a wrought-iron lattice tower in Paris."),
]

QUERY = "Who invented the C language?"


def show(label: str, hits: list[EmbeddingHit]) -> None:
    print(label)
    for rank, hit in enumerate(hits, start=1):
        print(f"  {rank}. {hit.distance:+.4f}  {hit.metadata['text']}")

## Embed the corpus

Metadata rides along with the vector and comes back on every hit — that is how
RAGU recovers entity and chunk payloads after a search.

In [ ]:
client = CachedAsyncOpenAI(
    base_url=os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1"),
    api_key=os.environ["OPENAI_API_KEY"],
    rate_max_simultaneous=10,
    rate_max_per_minute=100,
)
embedder = EmbedderOpenAI(client=client, model_name=os.environ["EMBEDDER_MODEL_NAME"])
await embedder.initialize()

texts = [text for _, text in DOCUMENTS]
vectors = await embedder.batch_embed_text(texts, desc="Embedding")
query_vector = np.array(await embedder.embed_text(QUERY))

points = [
    Point(id=doc_id, dense_embedding=np.array(vector), metadata={"text": text})
    for (doc_id, text), vector in zip(DOCUMENTS, vectors)
]

## NanoVectorDBStorage — dense only, JSON-backed

The default cosine threshold is `0.2`; dropping it shows every neighbour rather
than silently filtering the weak ones. Nano holds everything in memory until
`index_done_callback()` writes the file.

In [ ]:
workdir = Path(tempfile.mkdtemp(prefix="ragu_vector_adapters_"))

nano = NanoVectorDBStorage(
    embedding_dim=embedder.dim,
    storage_folder=str(workdir),
    filename="nano.json",
    score_threshold=0.0,
)
await nano.upsert(points)
await nano.index_done_callback()

show(f"query: {QUERY!r}", (await nano.query([Point(dense_embedding=query_vector)], top_k=3))[0])

Reads are index-aligned: a missing id yields `None` in its own slot rather than
shortening the list.

In [ ]:
print(f"ids:      {sorted(await nano.get_all_ids())}")
print(f"payloads: {await nano.get_payloads_by_ids(['doc-1', 'missing'])}")

await nano.delete(["doc-4"])
await nano.index_done_callback()
print(f"after delete: {sorted(await nano.get_all_ids())}")

await nano.close()
shutil.rmtree(workdir)

## QdrantVectorDBStorage — dense only, in-memory

`location=":memory:"` needs no server and no disk. Swap it for
`location="http://localhost:6333"` to talk to a real one, or drop it entirely to
get a local on-disk collection under `storage_folder`.

In [ ]:
dense = QdrantVectorDBStorage(
    embedding_dim=embedder.dim,
    location=":memory:",
    collection_name="adapters_dense",
)
await dense.upsert(points)

show(f"query: {QUERY!r}", (await dense.query([Point(dense_embedding=query_vector)], top_k=3))[0])

Batched search — one round trip for many queries, which is what the search engines
rely on when answering a batch of questions.

In [ ]:
queries = [QUERY, "Which language did Guido van Rossum design?"]
query_vectors = await embedder.batch_embed_text(queries)

batched = await dense.query(
    [Point(dense_embedding=np.array(vector)) for vector in query_vectors],
    top_k=2,
)
for query, hits in zip(queries, batched):
    show(f"\n{query!r}", hits)

await dense.close()

## QdrantVectorDBStorage — dense + BM25 (hybrid)

`sparse_type` declares the sparse vector field on the collection. Without it,
upserting a `Point` that carries a sparse embedding raises.

BM25 distinguishes indexing from querying: `embed_document` builds the
document-side term weights, `embed_query` the query-side ones.

In [ ]:
hybrid = QdrantVectorDBStorage(
    embedding_dim=embedder.dim,
    location=":memory:",
    collection_name="adapters_hybrid",
    sparse_type="bm25",
)

sparse_embedder = BM25()
sparse_vectors = sparse_embedder.embed_document(texts)
query_sparse = sparse_embedder.embed_query([QUERY])[0]

await hybrid.upsert([
    Point(
        id=point.id,
        dense_embedding=point.dense_embedding,
        sparse_embedding=sparse,
        metadata=point.metadata,
    )
    for point, sparse in zip(points, sparse_vectors)
])

A query carrying both vectors is fused server-side with reciprocal-rank fusion;
a query carrying only a dense one stays dense-only. Compare the two orderings —
BM25 pulls up the document that names the query terms literally.

In [ ]:
show("dense only", (await hybrid.query([Point(dense_embedding=query_vector)], top_k=3))[0])

show(
    "\nhybrid (dense + BM25, RRF fusion)",
    (await hybrid.query(
        [Point(dense_embedding=query_vector, sparse_embedding=query_sparse)],
        top_k=3,
    ))[0],
)

await hybrid.close()